In [4]:
import torch
import transformer_lens
import sae_lens
import pyvene
import transformers
from importlib.metadata import version

print("torch            :", torch.__version__)
print("transformer_lens :", version("transformer_lens"))
print("sae_lens         :", version("sae_lens"))
print("pyvene           :", version("pyvene"))
print("transformers     :", version("transformers"))
print("\nCUDA available:", torch.cuda.is_available())
print("Device we will use:", "cuda" if torch.cuda.is_available() else "cpu  (fine for testing)")

torch            : 2.12.0+cpu
transformer_lens : 3.2.1
sae_lens         : 6.43.0
pyvene           : 0.1.8
transformers     : 5.8.1

CUDA available: False
Device we will use: cpu  (fine for testing)


This downloads GPT-2 Small (~500 MB) the first time and caches it locally. You'll see a progress bar. Subsequent runs load from cache instantly.

In [5]:
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2", center_unembed=True, center_writing_weights=True)
model.eval()
print("Model loaded:", model.cfg.model_name)
print("Layers:", model.cfg.n_layers)
print("Heads per layer:", model.cfg.n_heads)
print("d_model:", model.cfg.d_model)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
Model loaded: gpt2
Layers: 12
Heads per layer: 12
d_model: 768


 Run a forward pass on a toy input

In [6]:
toy_prompt = "The Eiffel Tower is located in"

tokens = model.to_tokens(toy_prompt)
print("Token IDs:", tokens)
print("Tokens    :", model.to_str_tokens(toy_prompt))

logits = model(tokens)
top5 = logits[0, -1].topk(5)

print("\nTop 5 next-token predictions:")
for val, idx in zip(top5.values, top5.indices):
    print(f"  {model.to_single_str_token(idx.item())!r:12}  logit: {val.item():.2f}")

Token IDs: tensor([[50256,   464,   412,   733,   417,  8765,   318,  5140,   287]])
Tokens    : ['<|endoftext|>', 'The', ' E', 'iff', 'el', ' Tower', ' is', ' located', ' in']

Top 5 next-token predictions:
  ' the'        logit: 15.49
  ' a'          logit: 13.75
  ' central'    logit: 13.63
  ' London'     logit: 13.56
  ' downtown'   logit: 13.20


Cache activations with hooks (core MI tool)
This is the mechanism you will use constantly throughout the project.

In [7]:
tokens = model.to_tokens(toy_prompt)

# Run the model and cache every intermediate activation
logits, cache = model.run_with_cache(tokens)

print("Keys in the activation cache (first 10):")
for k in list(cache.keys())[:10]:
    print(" ", k)

# Inspect one: residual stream after layer 5
resid = cache["resid_post", 5]
print(f"\nresid_post at layer 5 — shape: {resid.shape}")
# shape is [batch, sequence_position, d_model]

Keys in the activation cache (first 10):
  hook_embed
  hook_pos_embed
  blocks.0.hook_resid_pre
  blocks.0.ln1.hook_scale
  blocks.0.ln1.hook_normalized
  blocks.0.attn.hook_q
  blocks.0.attn.hook_k
  blocks.0.attn.hook_v
  blocks.0.attn.hook_attn_scores
  blocks.0.attn.hook_pattern

resid_post at layer 5 — shape: torch.Size([1, 9, 768])


Load a pre-trained SAE for GPT-2 Small

sae_lens hosts SAEs trained on GPT-2 Small's residual stream. We load the one for layer 8 (a well-studied layer in the IOI circuit).

In [11]:
from sae_lens import SAE

sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.8.hook_resid_pre",
)
sae.eval()

print("SAE loaded for hook point:", sae.cfg.metadata["hook_name"])
print("Input dimension (d_model):", sae.cfg.d_in)
print("Dictionary size (features):", sae.cfg.d_sae)

SAE loaded for hook point: blocks.8.hook_resid_pre
Input dimension (d_model): 768
Dictionary size (features): 24576


Encode activations through the SAE

In [12]:
# Get the residual stream at layer 8 position for our toy prompt
resid_layer8 = cache["blocks.8.hook_resid_pre"] # [batch, seq, d_model]

# Encode through the SAE — get sparse feature activations
with torch.no_grad():
    feature_acts = sae.encode(resid_layer8) # [batch, seq, d_sae]
    resid_reconstructed = sae.decode(feature_acts)

last_pos = feature_acts[0, -1] # features at the final token position
active = (last_pos > 0).sum().item()

print(f"Features active at final token: {active} / {sae.cfg.d_sae}")
print(f"Reconstruction error (L2): {(resid_layer8[0,-1] - resid_reconstructed[0,-1]).norm().item():.4f}")

# Show top 5 most active features
top5_feats = last_pos.topk(5)
print("\nTop 5 active features:")
for val, idx in zip(top5_feats.values, top5_feats.indices):
    print(f"  feature {idx.item():6d}   activation: {val.item():.3f}")

Features active at final token: 57 / 24576
Reconstruction error (L2): 18.8828

Top 5 active features:
  feature   3076   activation: 13.771
  feature    313   activation: 12.763
  feature  19288   activation: 10.593
  feature  10092   activation: 5.331
  feature  24181   activation: 5.227


Basic causal intervention with pyvene

In [14]:
prompt_a = "The Eiffel Tower is located in"
prompt_b = "The Colosseum is located in"

tokens_a = model.to_tokens(prompt_a)
tokens_b = model.to_tokens(prompt_b)

# Get activations from prompt A
_, cache_a = model.run_with_cache(tokens_a)

# Base prediction on prompt B (no intervention)
base_logits = model(tokens_b)
base_pred = model.to_single_str_token(base_logits[0, -1].argmax().item())

# Patch layer 8 residual stream of prompt B with prompt A's activations
def patch_hook(value, hook):
    value[:, -1, :] = cache_a["blocks.8.hook_resid_pre"][:, -1, :]
    return value

patched_logits = model.run_with_hooks(
    tokens_b,
    fwd_hooks=[("blocks.8.hook_resid_pre", patch_hook)]
)
patched_pred = model.to_single_str_token(patched_logits[0, -1].argmax().item())

print(f"Base prediction    (no patch): '{base_pred}'")
print(f"Patched prediction (layer 8 residual from prompt A): '{patched_pred}'")
print("\nTransformerLens hook-based intervention ran without errors ✓")

Base prediction    (no patch): ' the'
Patched prediction (layer 8 residual from prompt A): ' the'

TransformerLens hook-based intervention ran without errors ✓
